# 00 — Inspect Pages

Load and validate `data/input/pages.json` before any LLM calls.

In [ ]:
from pathlib import Path
import pandas as pd

import sys
from pathlib import Path

_root = Path.cwd().resolve()
if _root.name == "notebooks":
    _root = _root.parent
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

from src.nb_utils import setup_project_root
from src.load_pages import (
    DEFAULT_INPUT,
    INSPECTION_KEYWORDS,
    keyword_flags,
    load_corpus,
    sort_pages_chronologically,
)

ROOT = setup_project_root()
print(f"Project root: {ROOT}")
print(f"Input file: {DEFAULT_INPUT}")

In [ ]:
corpus = load_corpus()
assert len(corpus.pages) == 10, f"Expected 10 pages, got {len(corpus.pages)}"
print(f"case_id: {corpus.case_id}")
print(f"date_of_loss: {corpus.date_of_loss}")

In [ ]:
rows = []
for p in sort_pages_chronologically(corpus.pages):
    flags = keyword_flags(p.text, INSPECTION_KEYWORDS)
    rows.append({
        "page": p.page_number,
        "chars": len(p.text),
        "date_of_service": p.date_of_service or "",
        "document_type": p.document_type or "",
        "keywords": ", ".join(flags) if flags else "",
        "preview": p.text[:120].replace("\n", " ") + "...",
    })

df = pd.DataFrame(rows)
df

In [ ]:
missing_dos = [p.page_number for p in corpus.pages if not p.date_of_service]
if missing_dos:
    print(f"Pages without date_of_service metadata: {missing_dos}")
    print("Agents will still run using page_number only.")
else:
    print("All pages have date_of_service metadata.")